# 07 · SpamAssassin ham — diagnostic annotation

**Цель:** диагностический аудит подвыборки SpamAssassin ham для понимания реальной структуры legitimate email корпуса.  
Результаты используются для проектирования prompt family `legitimate_email` (§5.9, §6.7 `dataset_design_final.md`).

**Выход:** `data/interim/annotated/spamassassin_ham_annotated.jsonl`

**Стратифицированная выборка:** 320 писем из 3 900 ham-строк, стратификация по `archive` × `wc_bin`.

---

## Категории (`scenario_family`)

| Категория | Описание |
|---|---|
| `business_email` | Рабочая коммуникация — встречи, проекты, внутренняя координация, деловые решения, взаимодействие с коллегами |
| `personal_email` | Личная или социальная коммуникация — семья, друзья, личные планы, частные дела |
| `service_or_system_email` | Автоматические сообщения от сервисов — системные уведомления, newsletters, mailing list, password reset, account update, subscription |
| `informational_notification_email` | Информационные обновления и объявления — новости, уведомления о событиях, организационные бюллетени (личные или профессиональные) |
| `mixed_or_unclear` | Явно охватывает несколько категорий, слишком неоднозначно или слишком короткое/повреждённое для классификации |

---

Аннотация через OpenRouter (`openai/gpt-4o-mini`). Resumable: уже аннотированные записи пропускаются через кэш по md5(text).


In [1]:
RANDOM_SEED = 42
N_SA_HAM = 320
BATCH_SIZE = 12
SLEEP_SEC = 2.5
MAX_NEW_THIS_RUN = 320   # full sample in one pass; re-runs are no-ops (cache skips already annotated)
ANNOTATION_MODEL = "openai/gpt-4o-mini"



In [2]:
import importlib.util, json, os, time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, APIError, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm.auto import tqdm

def _find_v2_root() -> Path:
    c = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", "."))).resolve()
    for p in [c, *c.parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "notebooks").exists(): return p
    raise RuntimeError("v2 root")
V2_ROOT = _find_v2_root()
RAW_DIR = V2_ROOT / "data" / "raw" / "collected"
INTERIM = V2_ROOT / "data" / "interim" / "annotated"
BLOCK_AB = INTERIM / "block_ab"
SAMP_DIR = BLOCK_AB / "samples"
OUT_FLAT = INTERIM / "spamassassin_ham_annotated.jsonl"
spec = importlib.util.spec_from_file_location("_ann_common", V2_ROOT / "notebooks" / "02_dataset_design" / "_ann_common.py")
ac = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ac)
load_dotenv(V2_ROOT / ".env" if (V2_ROOT / ".env").exists() else None)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])



In [3]:
sa_all = ac.load_jsonl(RAW_DIR / "spamassassin.jsonl")
sa_ham = [r for r in sa_all if r.get("label") == "ham"]
def build():
    rows = [{**r, "_wc": ac.wc_bin(ac.wc(r.get("text") or ""))} for r in sa_ham]
    df = pd.DataFrame(rows)
    picked = ac.stratified_sample_df(df, N_SA_HAM, ["archive", "_wc"], RANDOM_SEED + 13)
    return picked.drop(columns=["_wc"]).to_dict("records")
sample = ac.ensure_sample(SAMP_DIR / "sa_ham_stratified.jsonl", build)
print(len(sample))



320


In [4]:
SYSTEM_PROMPT = """\
You are annotating legitimate (non-spam) emails for a research dataset on LLM-generated text \
detection in anti-fraud systems.

Classify each email into exactly one category based on its primary intent:

- business_email: professional work communication — meetings, projects, internal coordination, \
business decisions, colleague interaction, work logistics
- personal_email: personal or social communication — family, friends, personal plans, private matters
- service_or_system_email: automated messages from online services — system alerts, newsletter, \
mailing list, password reset, account update, subscription notification
- informational_notification_email: informational updates and announcements — news, event \
notifications, organizational bulletins (could be personal or professional)
- mixed_or_unclear: clearly spans multiple categories, too ambiguous, or too short/garbled \
to classify reliably

Return strict JSON only — no explanation:
{"category": "<one of the five labels>", "confidence": "high|medium|low"}"""


def wrap(r):
    s = (r.get("subject") or "").strip()
    b = (r.get("text") or "").strip()
    user_content = f"Subject: {s}\n\n{b}" if s else b
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


OK = {"business_email", "personal_email", "service_or_system_email",
      "informational_notification_email", "mixed_or_unclear"}


def val(d):
    c = d.get("category", "mixed_or_unclear")
    if c not in OK:
        c = "mixed_or_unclear"
    conf = d.get("confidence", "low")
    if conf not in {"high", "medium", "low"}:
        conf = "low"
    return {"category": c, "confidence": conf}


@retry(retry=retry_if_exception_type((RateLimitError, APIError)),
       wait=wait_exponential(multiplier=2, min=2, max=45), stop=stop_after_attempt(6))
def call_json(msgs):
    return json.loads(client.chat.completions.create(
        model=ANNOTATION_MODEL, messages=msgs,
        response_format={"type": "json_object"}, temperature=0, max_tokens=128,
    ).choices[0].message.content)



In [5]:
idx = ac.load_flat_annotation_index(OUT_FLAT)
pending = [r for r in sample if ac.md5_text_key(r.get("text","")) not in idx]
batch_i = 0
for r in tqdm(pending[:MAX_NEW_THIS_RUN], desc="sa_ham"):
    try:
        a = val(call_json(wrap(r)))
    except Exception as e:
        a = val({}); a["_error"] = str(e)[:200]
    ts = datetime.now(timezone.utc).isoformat()
    ex = {"_error": a["_error"]} if "_error" in a else None
    flat = ac.make_flat_record(r, scenario_family=a["category"], annotation_confidence=a["confidence"], annotation_model=ANNOTATION_MODEL, annotated_at=ts, extra=ex)
    ac.append_jsonl(OUT_FLAT, flat)
    batch_i += 1
    if batch_i >= BATCH_SIZE:
        batch_i = 0
        time.sleep(SLEEP_SEC)
print(len(ac.load_flat_annotation_index(OUT_FLAT)))



sa_ham:   0%|          | 0/320 [00:00<?, ?it/s]

317


In [6]:
# ── Diagnostic summary ──────────────────────────────────────────────────────
# Run this cell after the annotation loop to inspect the audit results.
# The category distribution and sample texts are the primary output used
# for designing the `legitimate_email` prompt family (§6.7 dataset_design_final.md).

check = pd.DataFrame(ac.load_jsonl(OUT_FLAT))
if check.empty:
    print("No annotations yet — run the annotation loop first.")
else:
    print(f"Annotated: {len(check)} / target {N_SA_HAM}")

    print("\n--- category distribution ---")
    dist = check["scenario_family"].value_counts()
    for cat, cnt in dist.items():
        print(f"  {cat:45s}  {cnt:4d}  ({cnt / len(check) * 100:.1f}%)")

    print("\n--- confidence distribution ---")
    print(check["annotation_confidence"].value_counts().to_string())

    errors = check.get("_error", pd.Series(dtype=str)).dropna()
    if len(errors):
        print(f"\n--- annotation errors: {len(errors)} ---")

    print("\n--- 2 samples per category ---")
    for cat in dist.index:
        print(f"\n== {cat} ==")
        subset = check.loc[check["scenario_family"] == cat, ["subject", "text"]].dropna(subset=["text"])
        for _, row in subset.head(2).iterrows():
            subj = str(row.get("subject") or "").strip()
            body = str(row.get("text") or "").strip()
            if subj:
                print(f"  Subject: {subj[:100]}")
            print(f"  {body[:250]}")
            print()

Annotated: 320 / target 320

--- category distribution ---
  mixed_or_unclear                                157  (49.1%)
  business_email                                   58  (18.1%)
  informational_notification_email                 55  (17.2%)
  personal_email                                   29  (9.1%)
  service_or_system_email                          21  (6.6%)

--- confidence distribution ---
annotation_confidence
medium    184
high      133
low         3

--- 2 samples per category ---

== mixed_or_unclear ==
  Subject: [Spambayes] Current histograms
  >>> Tim Peters wrote
> We've not only reduced the f-p and f-n rates in my test runs, we've also
> made the score distributions substantially sharper.  This is bad news for
> Greg, because the non-existent "middle ground" is becoming even less
> exist

  Subject: RE: <nettime> The War Prayer
  I'm sure Patton used it.

I'm all for using it in the coming war with Iraq.

Yet I'd be queasy about doing it in the Philippines circa 19